# 6. Sankey: patient -> leading-edge gene -> pathway convergence

Per phenotype: literature/DB-validated group-level GSEA pathways (`Benchmark/PathwayCuration/<phenotype>.md`,
see `SUMMARY.md`) as the pathway tier, their GSEA leading-edge genes as the gene tier, and per-sample
normative Z-scores (`|Z|>1.96`) as patient-gene edges. Data source: `run_leading_edge_pattern.py` output
(`leading_edge_pattern_data.pkl`/`_summary.csv`) -- this notebook only visualizes, it does not recompute.

In [ ]:
import pickle
import re
import sys
from pathlib import Path

import cairosvg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scanpy as sc

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.SignalTrendAnalysis.sankey_helpers import strip_reactome_code
from MixedEffectsModeling.SignalTrendAnalysis.run_leading_edge_pattern import CURDIR, SLUG_MAP, Z_THRESH, parse_curation

PCDIR = config.PATHWAY_CONV_DIR  # sig.pkl/universe.pkl live in PerSamplePathwayAnalysis, not here
FIGDIR = CURDIR / 'Figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

heat_data = pickle.load(open(CURDIR / 'leading_edge_pattern_data.pkl', 'rb'))
summary = pd.read_csv(CURDIR / 'leading_edge_pattern_summary.csv')

In [ ]:
DEFAULT_PATIENT_COLOR = '#737373'
SEVERITY_COL = 'Stage/Condition'
SEVERITY_COLORS = {'PDAC': '#d55e00', 'IPMN': '#0072b2', 'Islet Cell Tumor': '#009e73'}
DISPLAY_NAME = {
    'Tuberculosis': 'Tuberculosis', 'Pancreatitis': 'Pancreatitis', 'Pancreatic_Cancer': 'Pancreatic Cancer',
    'Pre-eclampsia': 'Pre-eclampsia', 'Colorectal_Cancer': 'Colorectal Cancer', 'Lung_Cancer': 'Lung Cancer',
    'Esophagus_Cancer': 'Esophagus Cancer', 'Stomach_Cancer': 'Stomach Cancer',
    'Liver_Cancer_Roskams-Hieter': 'Liver Cancer (Roskams-Hieter)', 'Liver_Cancer_Chen': 'Liver Cancer (Chen)',
}

MAX_GENES_PER_PATHWAY = 12
HISTONE_FAMILY_CAP = 2
histone_rx = re.compile(r'^(H1|H2A|H2B|H3|H4)')


def gene_family(sym):
    m = histone_rx.match(sym)
    return m.group(1) if m else sym


def hex_to_rgba(hex_str, a):
    r, g, b = (int(hex_str.lstrip('#')[k:k + 2], 16) for k in (0, 2, 4))
    return f'rgba({r},{g},{b},{a})'


def build_sankey(stem):
    gsea_file, terms = parse_curation(CURDIR / f'{stem}.md')
    slug = SLUG_MAP[stem]
    pdir = PCDIR / slug
    d = pickle.load(open(pdir / 'sig.pkl', 'rb'))
    names_c = d['names_c']
    disp_name = DISPLAY_NAME[stem]

    if stem == 'Pancreatic_Cancer':
        severity = sc.read_h5ad(config.H5AD_PATH, backed='r').obs[SEVERITY_COL].reindex(names_c).values
        pat_color = lambda i: SEVERITY_COLORS.get(severity[i], DEFAULT_PATIENT_COLOR)
        pat_label = lambda i: f'{disp_name} {i + 1}' + (f' ({severity[i]})' if pd.notna(severity[i]) else '')
    else:
        pat_color = lambda i: DEFAULT_PATIENT_COLOR
        pat_label = lambda i: f'{disp_name} {i + 1}'

    cmap = plt.get_cmap('tab20')
    PATH_COLORS = [f'#{"".join(f"{int(c * 255):02x}" for c in cmap(ti % 20)[:3])}' for ti in range(len(terms))]

    # per pathway: rank leading-edge genes by total |Z| among |Z|>Z_THRESH hits, cap for readability
    pathway_gene_edges = {ti: {} for ti in range(len(terms))}
    for ti, term in enumerate(terms):
        key = (stem, term)
        if key not in heat_data:
            continue
        Zsub, genes = heat_data[key]
        hit = np.abs(Zsub) > Z_THRESH
        if not hit.any():
            continue
        strength = np.where(hit, np.abs(Zsub), 0).sum(axis=0)
        deg = hit.sum(axis=0)
        order = np.lexsort((-strength, -deg))
        picked, family_count = [], {}
        for gc in order:
            if strength[gc] == 0:
                continue
            fam = gene_family(genes[gc])
            if family_count.get(fam, 0) >= HISTONE_FAMILY_CAP:
                continue
            picked.append(gc)
            family_count[fam] = family_count.get(fam, 0) + 1
            if len(picked) == MAX_GENES_PER_PATHWAY:
                break
        for gc in picked:
            edges = {i: Zsub[i, gc] for i in range(len(names_c)) if hit[i, gc]}
            if edges:
                pathway_gene_edges[ti][genes[gc]] = edges

    gene_strength = {}
    for ti, edges_by_gene in pathway_gene_edges.items():
        for gname, edges in edges_by_gene.items():
            gene_strength.setdefault(gname, {})[ti] = sum(abs(z) for z in edges.values())
    if not gene_strength:
        print(f'{stem}: no gene edges survived Z_THRESH, skipping')
        return

    gene_primary_ti = {g: max(dd, key=dd.get) for g, dd in gene_strength.items()}
    displayed_genes = sorted(gene_strength, key=lambda g: (gene_primary_ti[g], -gene_strength[g][gene_primary_ti[g]]))

    pat_gene_z = {}
    for edges_by_gene in pathway_gene_edges.values():
        for gname, edges in edges_by_gene.items():
            for i, z in edges.items():
                pat_gene_z[(i, gname)] = abs(z)
    displayed_pats = sorted({i for i, _ in pat_gene_z})
    used_ti = sorted({ti for ti, e in pathway_gene_edges.items() if e})

    pat_pos = {i: k for k, i in enumerate(displayed_pats)}
    gene_pos = {g: k for k, g in enumerate(displayed_genes)}
    path_pos = {ti: k for k, ti in enumerate(used_ti)}
    n_pat_d, n_gene_d = len(displayed_pats), len(displayed_genes)
    n_pat_total = len(names_c)

    node_labels = ([pat_label(i) for i in displayed_pats] + displayed_genes
                  + [strip_reactome_code(terms[ti]) for ti in used_ti])
    node_colors = ([pat_color(i) for i in displayed_pats] + [PATH_COLORS[gene_primary_ti[g]] for g in displayed_genes]
                  + [PATH_COLORS[ti] for ti in used_ti])

    src, tgt, val, link_col = [], [], [], []
    for (i, gname), z in pat_gene_z.items():
        src.append(pat_pos[i])
        tgt.append(n_pat_d + gene_pos[gname])
        val.append(z)
        link_col.append(hex_to_rgba(pat_color(i), 0.35))
    for ti, edges_by_gene in pathway_gene_edges.items():
        if ti not in path_pos:
            continue
        for gname, edges in edges_by_gene.items():
            src.append(n_pat_d + gene_pos[gname])
            tgt.append(n_pat_d + n_gene_d + path_pos[ti])
            val.append(sum(abs(z) for z in edges.values()))
            link_col.append(hex_to_rgba(PATH_COLORS[ti], 0.45))

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=node_labels, color=node_colors, pad=3, thickness=10, line=dict(width=0)),
        link=dict(source=src, target=tgt, value=val, color=link_col),
    ))
    fig.update_layout(width=1000, height=max(700, (n_pat_d + n_gene_d) * 13), font_size=11,
                      title=f'{disp_name}: patient -> leading-edge gene -> pathway ({n_pat_d}/{n_pat_total} patients shown)')

    svg = fig.to_image(format='svg').decode('utf-8')
    svg = re.sub(r'text-shadow:[^;"]*;', '', svg)
    out_path = FIGDIR / f'{stem}_sankey.png'
    cairosvg.svg2png(bytestring=svg.encode('utf-8'), write_to=str(out_path), scale=2)
    print(f'{disp_name}: {n_pat_d}/{n_pat_total} patients shown, {n_gene_d} genes, '
          f'{len(used_ti)}/{len(terms)} pathways -> {out_path}')
    return fig

In [ ]:
for stem in SLUG_MAP:
    fig = build_sankey(stem)
    if fig is not None:
        fig.show()